In [1]:
import pandas as pd
import scipy as sp
import numpy as np
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import recall_at_k
from lightfm.evaluation import auc_score
from lightfm import LightFM
import matplotlib.pyplot as plt
import pickle
import collections
import seaborn as sns
import gzip
import textwrap
import os
from datetime import date

**Create trait vs SNP matrix**

In [2]:
#load data
data = []
with open("/home/nasi4/proj/GWAS/UKBB_201807/results/all.UKBB.hg38.sig.phenotype.bed")as f:
    for line in f:
        data.append(line.strip().split())

#combine label elements
for row in data:
    row.append(' '.join(row[5:]))
    del row[5:-1]
    
#create dataframe
df = pd.DataFrame(data, columns = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'p-value', 'Trait'])

df = df[df['p-value']!='0.00000e+00']
test_list = df['p-value'].to_list()
test_list = list(map(float, test_list))
print(max(test_list))

#calculating the negative log of the p-value, setting infinite values to the max non-infinite value observed, and
#normalizing by max -log(p-value) score
pval_mlog = -np.log(list(map(float, list(df['p-value']))))

inf_mask = np.ma.array(pval_mlog, mask=~np.isfinite(pval_mlog))
max_ind = np.ma.argmax(inf_mask, fill_value=-np.inf)
print(max_ind)
max_val = pval_mlog[max_ind]
print(max_val)

for ind in range(len(pval_mlog)):
    if pval_mlog[ind] > pval_mlog[max_ind]:
        pval_mlog[ind] = pval_mlog[max_ind]
        
df['pval_mlog'] = pval_mlog/pval_mlog[max_ind]

df[(df['Trait']=='Weight')&(df['Reference SNP cluster ID']=='16:53812770_ATTTT_A')]

# Reference SNP cluster ID: rs11642015, chr16:53768582-53768583, p-value = 1.14192e-191

#creating df with SNP, trait, and p-value
ts_df = df[['Reference SNP cluster ID', 'Trait', 'pval_mlog']]

#finding unique traits and SNPs
SNP_values = np.unique(ts_df['Reference SNP cluster ID'].values)
trait_values = np.unique(ts_df['Trait'].values)

ts_df[(ts_df['Trait']=='Weight')&(ts_df['Reference SNP cluster ID']=='rs11642015')]

ts = ts_df.groupby(['Reference SNP cluster ID', 'Trait']).max()

#20211129 - changed from original way I did this. 
# I was averaging all -ln(p-values) from a given trait vs SNP combination
# now I am taking the max valeu
# ts = ts_df.groupby(['Reference SNP cluster ID', 'Trait']).mean()
ts = ts.reset_index()

ts['trait_ind'] = ts.groupby(['Trait']).ngroup()
ts['snp_ind'] = ts.groupby(['Reference SNP cluster ID']).ngroup()

columns = ts['snp_ind'].to_list()
rows = ts['trait_ind'].to_list()
pval_mlog_data = ts['pval_mlog'].to_list()

data_coo = sp.sparse.coo_matrix((pval_mlog_data, (rows, columns)), shape=(len(trait_values), len(SNP_values)))

5e-08
736764
743.7469247408213


**Load Model**

In [3]:
pickle_in = open('/home/ctrader/Hold/GWAS/gwas_model_e30_d10_reg00001_CT20211006.pickle', 'rb')
gwas_model_e30_d10_reg00001=pickle.load(pickle_in)

pickle_in = open('/home/ctrader/Hold/GWAS/gwas_data_e30_d10_reg00001_CT20211006.pickle', 'rb')
data_e30_d10_reg00001=pickle.load(pickle_in)

In [4]:
snp_b, snp_e = gwas_model_e30_d10_reg00001.get_item_representations()
snp_e = snp_e.T
trait_b, trait_e = gwas_model_e30_d10_reg00001.get_user_representations()

In [5]:
n_dimensions, n_snps = snp_e.shape
n_traits, n_dimensions = trait_e.shape

In [6]:
# gwas_model_e30_d10_reg00001.predict(1600, np.arange(n_snps))

In [7]:
predictions = ((np.dot(trait_e, snp_e) + snp_b).T + trait_b).T

In [8]:
predictions_sorted = np.argsort(-predictions, axis=1)

In [11]:
predictions_sorted_100 = predictions_sorted[:, :100]

In [13]:
def generate_100_recs(model):
    #retrieving the trait and SNP biases and embeddings for calculating prediction scores
    snp_b, snp_e = model.get_item_representations()
    snp_e = snp_e.T
    trait_b, trait_e = model.get_user_representations()
    n_dimensions, n_snps = snp_e.shape
    n_traits, n_dimensions = trait_e.shape
    #calculating predictions scores for each trait-SNP interaction
    predictions = ((np.dot(trait_e, snp_e) + snp_b).T + trait_b).T
    #sorting prediction scores to get the ordered SNP recommendations for each trait
    predictions_sorted = np.argsort(-predictions, axis=1)
    #taking the first 100 SNP recommendations for each trait
    predictions_sorted_100 = predictions_sorted[:, :100]
    return predictions_sorted_100